In [8]:
import pandas as pd
df=pd.read_csv('cleaned_data.csv')

print(df.isnull().sum())
print(df.duplicated().sum())
df.sample(10)




answers    0
label      0
dtype: int64
0


,answers,label
23946,"['In the early days of military aviation, plan...",1
27237,"[""HOAs, or homeowners associations, are organi...",1
23315,"[""There are a few reasons why a Valve or Bethe...",1
5538,"[""Let 's say you have a point that emits light...",0
42872,['It can be helpful to hire an accountant to h...,1
25223,"['Color blindness, also known as color vision ...",1
9874,"['it \'s short for the latin "" libra "" - which...",0
12002,"[""Rotating objects , like wheels , have a prop...",0
32162,['It\'s not accurate to say that liberals are ...,1
8789,"[""Because the surface is non porous , so the d...",0


In [9]:
import ast

def clean_text(x):
    if isinstance(x, list):
        return " ".join(map(str, x))

    if isinstance(x, str):
        try:
            parsed = ast.literal_eval(x)

            if isinstance(parsed, list):
                return " ".join(map(str, parsed))
            else:
                return str(parsed)

        except:
            return x   # IMPORTANT: fallback safe

    return str(x)



In [10]:
df['answers'] = df['answers'].str.lower()
df['answers']=df['answers'].apply(clean_text)

df.head()

,answers,label
0,"basically there are many categories of "" best ...",0
1,salt is good for not dying in car crashes and ...,0
2,the way it works is that old tv stations got a...,0
3,you ca n't just go around assassinating the le...,0
4,wanting to kill the shit out of germans drives...,0


In [11]:
df_feat = df.copy()
df_feat.sample(5)

,answers,label
16002,barefoot and pregnant is a phrase most commonl...,0
19176,generally speaking the lower credit score trum...,0
24433,the watergate scandal was a political scandal ...,1
20113,split transactions are indispensable to anybod...,0
40091,computer hardware refers to the physical compo...,1


In [12]:
import spacy
nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])
nlp.add_pipe("sentencizer")

In [13]:
def load_word_list(path):
    with open(path, "r", encoding="utf-8") as f:
        return set(line.strip().lower() for line in f if line.strip())

chat_words = load_word_list("chat_words.txt")
function_words = load_word_list("function_words.txt")
discourse_markers = load_word_list("discourse_markers.txt")

In [14]:
def split_sentences(text):
    doc = nlp(text)
    return [sent.text.strip() for sent in doc.sents if sent.text.strip()]

In [15]:
def extract_features(text):
    doc = nlp(text)

    tokens = [t.text.lower() for t in doc if not t.is_space]
    alpha_tokens = [t.text.lower() for t in doc if t.is_alpha]

    total_tokens = len(tokens)
    total_alpha = len(alpha_tokens)

    if total_tokens == 0 or total_alpha == 0:
        return {
            "chat_word_count": 0,
            "punct_count": 0,
            "function_count": 0,
            "discourse_count": 0,
            "spelling_error_ratio": 0,
            "ttr": 0,
            "function_word_ratio": 0,
            "discourse_ratio": 0,
            "sentence_length": 0,
            "avg_word_length": 0,
        }

    chat_word_count = sum(1 for t in tokens if t in chat_words)
    punct_count = sum(1 for t in doc if t.is_punct)

    function_count = sum(1 for t in tokens if t in function_words)
    discourse_count = sum(1 for t in tokens if t in discourse_markers)

    spelling_errors = sum(1 for t in doc if t.is_alpha and t.is_oov)

    return {
        "chat_word_count": chat_word_count,
        "punct_count": punct_count,
        "function_count": function_count,
        "discourse_count": discourse_count,
        "spelling_error_ratio": spelling_errors / total_alpha,
        "ttr": len(set(alpha_tokens)) / total_alpha,
        "function_word_ratio": function_count / total_tokens,
        "discourse_ratio": discourse_count / total_tokens,
        "sentence_length": total_tokens,
        "avg_word_length": sum(len(t) for t in alpha_tokens) / total_alpha,
    }

In [16]:
def aggregate_sentence_features(sentences):
    feats = [extract_features(s) for s in sentences]

    if len(feats) == 0:
        return {}

    df = pd.DataFrame(feats)

    return {
        "chat_word_count": df["chat_word_count"].sum(),
        "punct_count": df["punct_count"].sum(),

        "function_count": df["function_count"].sum(),
        "discourse_count": df["discourse_count"].sum(),

        "spelling_error_ratio": df["spelling_error_ratio"].mean(),
        "ttr": df["ttr"].mean(),
        "function_word_ratio": df["function_word_ratio"].mean(),
        "discourse_ratio": df["discourse_ratio"].mean(),

        "sentence_length_mean": df["sentence_length"].mean(),
        "sentence_length_std": df["sentence_length"].std(),

        "avg_word_length": df["avg_word_length"].mean(),
    }

In [17]:
def build_final_features(df):
    rows = []

    for _, row in df.iterrows():
        text = row["answers"]
        label = row["label"]

        # 1. split into sentences
        sentences = split_sentences(text)

        # 2. aggregate sentence-level features → document-level features
        features = aggregate_sentence_features(sentences)

        # 3. attach label
        features["label"] = label

        rows.append(features)

    return pd.DataFrame(rows)

In [18]:
#df_final = build_final_features(df_feat)

In [19]:
#import joblib
#joblib.dump(df_final, "final_features.pkl")

In [20]:
import joblib
df_feat = joblib.load("final_features.pkl")

In [21]:
df_feat.head()

,chat_word_count,punct_count,function_count,discourse_count,spelling_error_ratio,ttr,function_word_ratio,discourse_ratio,sentence_length_mean,sentence_length_std,avg_word_length,label
0,2.0,48.0,118.0,15.0,1.0,0.899103,0.442874,0.059134,24.272727,14.185139,4.373566,0
1,1.0,46.0,192.0,24.0,1.0,0.918157,0.458618,0.049649,20.750000,10.289877,4.537067,0
2,4.0,55.0,196.0,24.0,1.0,0.900015,0.391254,0.042689,36.923077,23.514044,4.643522,0
3,1.0,20.0,92.0,12.0,1.0,0.933176,0.415362,0.055521,22.111111,19.303137,4.923874,0
4,1.0,29.0,147.0,15.0,1.0,0.868949,0.512040,0.050100,31.666667,12.549900,4.692924,0


In [22]:
df_final = pd.concat(
    [
        df[["answers", "label"]].reset_index(drop=True),
        df_feat.drop(columns=["label"], errors="ignore").reset_index(drop=True)
    ],
    axis=1
)
df_final.head()

,answers,label,chat_word_count,punct_count,function_count,discourse_count,spelling_error_ratio,ttr,function_word_ratio,discourse_ratio,sentence_length_mean,sentence_length_std,avg_word_length
0,"basically there are many categories of "" best ...",0,2.0,48.0,118.0,15.0,1.0,0.899103,0.442874,0.059134,24.272727,14.185139,4.373566
1,salt is good for not dying in car crashes and ...,0,1.0,46.0,192.0,24.0,1.0,0.918157,0.458618,0.049649,20.750000,10.289877,4.537067
2,the way it works is that old tv stations got a...,0,4.0,55.0,196.0,24.0,1.0,0.900015,0.391254,0.042689,36.923077,23.514044,4.643522
3,you ca n't just go around assassinating the le...,0,1.0,20.0,92.0,12.0,1.0,0.933176,0.415362,0.055521,22.111111,19.303137,4.923874
4,wanting to kill the shit out of germans drives...,0,1.0,29.0,147.0,15.0,1.0,0.868949,0.512040,0.050100,31.666667,12.549900,4.692924
